# Midterm NLP

## Data Acquisition

Dalam step ini dilakukan pengumpulan url berita yang berkaitan dengan MBG berdasarkan beberapa keywords tertentu. Setelah pengumpulan url berita dilakukan scraping untuk mengambil content data.

### Import Library

In [21]:
import asyncio
import os
import random
from typing import Optional

import pandas as pd
from tqdm import tqdm
import urllib.parse
import feedparser
from playwright.async_api import Browser, async_playwright
from bs4 import BeautifulSoup

### RSS Fetch

#### Variables Defining

In [41]:
keyword_file = "keyword.txt"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "articles.csv")
max_concurrent = 10
show_progress = True

limit_per_keyword = 0  # Set to 0 for no limit

In [42]:
class RSSFetchError(Exception):
    pass


def read_keywords(filepath: str) -> list[str]:
    keywords = []
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    keywords.append(line)
        return keywords
    except Exception as e:
        raise RSSFetchError(f"Failed to read keywords from {filepath}: {e}") from e


def fetch_rss_feed(
    keyword: str,
    region: str = "ID",
    language: str = "id",
    limit: int | None = None,
) -> list[dict]:
    safe_keyword = urllib.parse.quote(keyword)
    rss_url = f"https://news.google.com/rss/search?q={safe_keyword}&hl={language}&gl={region}&ceid={region}:{language}"

    try:
        feed = feedparser.parse(rss_url)
    except Exception as e:
        raise RSSFetchError(f"Failed to fetch RSS for '{keyword}': {e}") from e

    entries = feed.entries if limit is None else feed.entries[:limit]
    results = []
    for entry in entries:
        source_title = ""
        source_href = ""
        if hasattr(entry, "source"):
            source_title = entry.source.get("title", "")
            source_href = entry.source.get("href", "")

        results.append(
            {
                "title": getattr(entry, "title", ""),
                "link": getattr(entry, "link", ""),
                "source": source_title,
                "source_url": source_href,
                "pub_date": getattr(entry, "published", ""),  # feedparser maps <pubDate> to entry.published
            }
        )

    return results


def fetch_all_feeds(
    keywords: list[str],
    limit_per_keyword: int | None = None,
    region: str = "ID",
    language: str = "id",
    progress_callback=None,
) -> list[dict]:
    all_entries = []
    for i, keyword in enumerate(keywords):
        if progress_callback:
            progress_callback(i, len(keywords), keyword)
        try:
            entries = fetch_rss_feed(
                keyword, region=region, language=language, limit=limit_per_keyword
            )
            for entry in entries:
                entry["keyword"] = keyword
            all_entries.extend(entries)
        except RSSFetchError as e:
            print(f"[rss_fetcher] Error for '{keyword}': {e}")
    return all_entries

In [43]:
keywords = read_keywords(keyword_file)
print(f"Loaded {len(keywords)} keywords:")
for kw in keywords:
    print(f"  - {kw}")

Loaded 9 keywords:
  - Makan Bergizi Gratis
  - Makan Siang Gratis
  - MBG
  - Program Makan Gratis Pemerintah
  - Badan Gizi Nasional
  - Satuan Pelayanan Pemenuhan Gizi
  - SPPG
  - Perpres Nomor 83 Tahun 2024
  - Perpres Nomor 115 Tahun 2025


In [44]:
# Step 1: Fetch RSS feeds
_limit = None if limit_per_keyword == 0 else limit_per_keyword

all_entries = []
for keyword in tqdm(keywords, desc="Fetching RSS feeds", disable=not show_progress):
    entries = fetch_rss_feed(keyword, limit=_limit)
    for entry in entries:
        entry["keyword"] = keyword
    all_entries.extend(entries)

print(f"\nTotal entries fetched: {len(all_entries)}")

# Deduplicate by Google redirect URL
seen_links = set()
unique_entries = []
for entry in all_entries:
    if entry["link"] not in seen_links:
        seen_links.add(entry["link"])
        unique_entries.append(entry)

print(f"Unique entries after dedup: {len(unique_entries)}")

Fetching RSS feeds: 100%|██████████| 9/9 [00:10<00:00,  1.13s/it]


Total entries fetched: 910
Unique entries after dedup: 797


### URL Resolution

In [45]:
class URLResolutionError(Exception):
    pass


class RateLimiter:
    def __init__(self, min_delay: float, max_delay: float):
        self.min_delay = min_delay
        self.max_delay = max_delay

    async def wait(self):
        delay = random.uniform(self.min_delay, self.max_delay)
        await asyncio.sleep(delay)


class ExponentialBackoff:
    def __init__(self, max_retries: int, base_delay: float = 1.0):
        self.max_retries = max_retries
        self.base_delay = base_delay

    async def execute_with_retry(self, func, *args, **kwargs):
        last_exc = None
        for attempt in range(self.max_retries):
            try:
                return await func(*args, **kwargs)
            except Exception as e:
                last_exc = e
                if attempt < self.max_retries - 1:
                    wait_time = (2 ** attempt) * self.base_delay + random.uniform(0, 1)
                    await asyncio.sleep(wait_time)
        raise URLResolutionError(f"All {self.max_retries} retries failed: {last_exc}") from last_exc


class URLResolver:
    def __init__(
        self,
        headless: bool = True,
        max_concurrent: int = 5,
        min_delay: float = 1.0,
        max_delay: float = 3.0,
        max_retries: int = 3,
        timeout: int = 30000,
    ):
        self.headless = headless
        self.max_concurrent = max_concurrent
        self.timeout = timeout
        self._rate_limiter = RateLimiter(min_delay, max_delay)
        self._backoff = ExponentialBackoff(max_retries)
        self._playwright = None
        self._browser: Optional[Browser] = None
        self._semaphore: Optional[asyncio.Semaphore] = None

    async def __aenter__(self) -> "URLResolver":
        self._playwright = await async_playwright().start()
        self._browser = await self._playwright.chromium.launch(headless=self.headless)
        self._semaphore = asyncio.Semaphore(self.max_concurrent)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self.close()

    async def close(self):
        if self._browser:
            await self._browser.close()
            self._browser = None
        if self._playwright:
            await self._playwright.stop()
            self._playwright = None

    async def _do_resolve(self, redirect_url: str) -> tuple[str, Optional[str]]:
        context = await self._browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        try:
            page = await context.new_page()
            await self._rate_limiter.wait()
            await page.goto(redirect_url, wait_until="domcontentloaded", timeout=self.timeout)
            # Google News uses JS to redirect — wait for the URL to leave news.google.com
            if "news.google.com" in page.url:
                try:
                    await page.wait_for_url(
                        lambda url: "news.google.com" not in url,
                        timeout=15000,
                    )
                except Exception:
                    pass  # URL didn't change; keep whatever we have
            final_url = page.url
            html_content = await page.content()
            return final_url, html_content
        finally:
            await context.close()

    async def resolve_url(self, redirect_url: str) -> tuple[str, Optional[str]]:
        async with self._semaphore:
            return await self._backoff.execute_with_retry(self._do_resolve, redirect_url)

    async def resolve_batch(
        self,
        urls: list[str],
        progress_callback=None,
    ) -> list[tuple[str, Optional[str]]]:
        completed = 0
        total = len(urls)
        results = [None] * total
        lock = asyncio.Lock()

        async def resolve_one(i: int, url: str):
            nonlocal completed
            try:
                result = await self.resolve_url(url)
            except URLResolutionError as e:
                result = (url, None)
                print(f"[url_resolver] Failed to resolve {url}: {e}")
            results[i] = result
            async with lock:
                completed += 1
                if progress_callback:
                    await progress_callback(total, completed, url)

        await asyncio.gather(*[resolve_one(i, url) for i, url in enumerate(urls)])
        return results

In [46]:
# Step 2: Resolve redirect URLs with Playwright
redirect_urls = [e["link"] for e in unique_entries]

with tqdm(total=len(redirect_urls), desc="Resolving URLs", disable=not show_progress) as pbar:
    async def _url_progress(total, completed, url):
        pbar.n = completed
        pbar.set_postfix_str(url[:50])
        pbar.refresh()

    async with URLResolver(max_concurrent=max_concurrent) as resolver:
        resolved = await resolver.resolve_batch(redirect_urls, _url_progress)

print(f"Resolved {len(resolved)} URLs")

Resolving URLs:  19%|█▉        | 154/797 [05:09<21:33,  2.01s/it, https://news.google.com/rss/articles/CBMi0gFBVV95c]

[url_resolver] Failed to resolve https://news.google.com/rss/articles/CBMi0gFBVV95cUxQNGE5N3dzejR2SllrZmFsV3NWMFRzX2E1WFJHV3JKLXZSYldBQnZKZXByalhYa1Q4ems4RDlZYWhJUXIyRWdSZ1B0ekJrZk5vd2Z3aERBTlk3UVVGQm5rLWw1VF8ydkdpTEZQZUZSd2YzQjN1aU1UaU8yV2hObEhpdDBxNXBzY29Lc0pYR1dSRGptR096SDlaem1yOGtwUG50NlpFaURiRFprOVV4Z0FOX0stanRIcUNCbEFsMVk2ZTBZRXVqN25NUkZIbl9OblFOZ1E?oc=5: All 3 retries failed: Page.content: Unable to retrieve content because the page is navigating and changing the content.


Resolving URLs:  20%|██        | 161/797 [05:31<21:50,  2.06s/it, https://news.google.com/rss/articles/CBMiqwFBVV95c]

[url_resolver] Failed to resolve https://news.google.com/rss/articles/CBMiqwFBVV95cUxQRm9vdHpiMUZnRzlBTk1rUXdfUWJyVGVHZzNDUHJBdGE5U2N6ZWduVTJ2NE1qWktFYkdYZWNvT2tjSEZDTFdzdHdWc1lzclZ6TjZWSU94a3FsbDBPcWhXNHhuWXJPZlBmenQ5UkVsWWdBeGZtM2F4Q3ZNdm43a3FmM2ZZN1NYbEEzSWhZLVJUY0RUTnJKckFhQ1Jhc1dqYlhUWFdBdGZFUHNhdjg?oc=5: All 3 retries failed: Page.content: Unable to retrieve content because the page is navigating and changing the content.


Resolving URLs:  21%|██        | 164/797 [05:37<21:40,  2.06s/it, https://news.google.com/rss/articles/CBMia0FVX3lxT]

...

[url_resolver] Failed to resolve https://news.google.com/rss/articles/CBMipwFBVV95cUxPaVYteVRFR3RRbjE0WWFrbWpqUmdvc0JVVGY5aTVlUjNKWUYxUDQ0Rk9SOHN3cE5PcFBFS2tObWVQNGE1djJ6R1dRNS1wV3gzMV9QYW0zSXZ3TWZ5MHJaV2RTSENtdnI3cVFoSVkzRTRJX1kwa2k1U1BWSUxRNFM4X2pjX2RYcTBCbTE3TElkOWEybjJmTzBqX3Y0Nk9LS3QtNXpnelc1UQ?oc=5: All 3 retries failed: Page.content: Unable to retrieve content because the page is navigating and changing the content.


Resolving URLs:  97%|█████████▋| 772/797 [25:15<00:49,  1.96s/it, https://news.google.com/rss/articles/CBMiwgFBVV95c]

[url_resolver] Failed to resolve https://news.google.com/rss/articles/CBMiwgFBVV95cUxOeWZPQXFMeEhZUWR5YXF3TXl3TnoyRlhWaVN0Q2pUdTFBMHdMVUF0YTEtc1pfczFybW1QZEdYb1BrMjZBWFkzYmNvNDdyWXB0Ykh4UmxxT251d0t0U3d4djItRE1ZbE1kRG9mUkw1VjJuLVVuQUppUk1xcEZMQVhQOU1uVWU3ZUVXMVNsVlI4YllBemYyS0p6VGhPR0FiM0U4dXN1NnR1ZWJFc0xXLWNGZkRNUnlIYjhvTGp4UXZYQUNLQQ?oc=5: All 3 retries failed: Page.content: Unable to retrieve content because the page is navigating and changing the content.


Resolving URLs: 100%|██████████| 797/797 [27:33<00:00,  2.07s/it, https://news.google.com/rss/articles/CBMif0FVX3lxT]

[url_resolver] Failed to resolve https://news.google.com/rss/articles/CBMif0FVX3lxTFBXVlhIQWlBdmY5emJDWm5GRkUzdkNQbXlqUVA3VG9GVGpGMnhQWFJkN1RkcF9iSnhKSUxzXzRENlM4T0J2ejV6VTJkczFXUmZ1T3VEbElKSDVQTjgwWDkySWNMc0U4SXFsWjAxTHhBRjh5WUZieWNNanZxdXU5NWM?oc=5: All 3 retries failed: Page.content: Unable to retrieve content because the page is navigating and changing the content.
Resolved 797 URLs


### Article Parsing

In [47]:
def parse_article_bs4(html_content: Optional[str], url: str) -> dict:
    if not html_content:
        return {"title": "", "text": "NO_CONTENT_EXTRACTED", "pub_date": None, "authors": []}

    try:
        soup = BeautifulSoup(html_content, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "iframe"]):
            tag.decompose()

        # Title
        title = ""
        h1 = soup.find("h1")
        if h1:
            title = h1.get_text(strip=True)
        elif soup.title:
            title = soup.title.get_text(strip=True)

        # Published date from common meta tags
        pub_date = None
        for attrs in [
            {"property": "article:published_time"},
            {"name": "article:published_time"},
            {"name": "publishedDate"},
            {"name": "date"},
            {"itemprop": "datePublished"},
        ]:
            meta = soup.find("meta", attrs)
            if meta and meta.get("content"):
                pub_date = meta["content"]
                break

        # Authors
        authors = []
        author_meta = soup.find("meta", {"name": "author"})
        if author_meta and author_meta.get("content"):
            authors.append(author_meta["content"])
        if not authors:
            for tag in soup.find_all(attrs={"rel": "author"}):
                text = tag.get_text(strip=True)
                if text:
                    authors.append(text)
        if not authors:
            for tag in soup.find_all(attrs={"itemprop": "author"}):
                text = tag.get_text(strip=True)
                if text:
                    authors.append(text)

        # Main text: prefer <article>, then <main>, fall back to <body>
        container = soup.find("article") or soup.find("main") or soup.find("body")
        paragraphs = container.find_all("p") if container else soup.find_all("p")
        text = "\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))

        return {
            "title": title,
            "text": text or "NO_CONTENT_EXTRACTED",
            "pub_date": pub_date,
            "authors": authors,
        }
    except Exception as e:
        return {"title": "", "text": f"FAILED_TO_PARSE: {e}", "pub_date": None, "authors": []}

In [48]:
# Step 3: Parse articles with BeautifulSoup
rows = []
skipped_url = 0
skipped_content = 0

parse_iter = tqdm(
    zip(unique_entries, resolved),
    total=len(unique_entries),
    desc="Parsing articles",
    disable=not show_progress,
)
for entry, (final_url, html_content) in parse_iter:
    if "news.google.com" in final_url:
        skipped_url += 1
        continue

    parsed = parse_article_bs4(html_content, final_url)

    if parsed["text"] in ("NO_CONTENT_EXTRACTED",) or parsed["text"].startswith("FAILED_TO_PARSE"):
        skipped_content += 1
        continue

    rows.append({
        "keyword": entry["keyword"],
        "judul": entry["title"],
        "tanggal": entry["pub_date"],
        "sumber": entry["source"],
        "source_url": entry["source_url"],
        "url_google": entry["link"],
        "url_resolved": final_url,
        "konten": parsed["text"],
        "authors": ", ".join(parsed["authors"]),
    })

print(f"Skipped {skipped_url} unresolved (still news.google.com)")
print(f"Skipped {skipped_content} with no extractable content")

# Step 4: Save to CSV
df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["url_resolved"])
df.to_csv(output_file, index=False)
print(f"\nDone! {len(df)} unique articles saved to {output_file}")
df

Parsing articles: 100%|██████████| 797/797 [00:28<00:00, 27.49it/s]

Skipped 9 unresolved (still news.google.com)
Skipped 188 with no extractable content

Done! 599 unique articles saved to output/articles.csv


,keyword,judul,tanggal,sumber,source_url,url_google,url_resolved,konten,authors
0,Makan Bergizi Gratis,Kampus diminta 'terlibat aktif' bangun dapur M...,"Fri, 01 May 2026 07:39:41 GMT",BBC,https://www.bbc.com,https://news.google.com/rss/articles/CBMiYEFVX...,https://www.bbc.com/indonesia/articles/c5y8jyg...,"Sumber gambar,ANTARA FOTO/Adeng Bustomi\nKeter...",
1,Makan Bergizi Gratis,"Dukung Kampus Punya Dapur MBG, Mendiktisaintek...","Sat, 02 May 2026 12:37:00 GMT",Medcom.id,https://www.medcom.id,https://news.google.com/rss/articles/CBMi1gFBV...,https://www.medcom.id/pendidikan/news-pendidik...,Ikuti media sosial medcom.id dan dapatkan berb...,medcom.id developer
2,Makan Bergizi Gratis,Mendikti Minta Kampus Lain Tiru Langkah Unhas ...,"Thu, 30 Apr 2026 03:03:43 GMT",Tempo.co,https://www.tempo.co,https://news.google.com/rss/articles/CBMioAFBV...,https://www.tempo.co/politik/mendikti-minta-ka...,"MENTERI Pendidikan Tinggi, Sains, dan Teknolog...",
3,Makan Bergizi Gratis,Program makan bergizi di Jambi sentuh 446.087 ...,"Sat, 02 May 2026 11:47:23 GMT",ANTARA News Jambi,https://jambi.antaranews.com,https://news.google.com/rss/articles/CBMipwFBV...,https://jambi.antaranews.com/berita/655860/pro...,Kota Jambi (ANTARA) - Program Makan Bergizi Gr...,ANTARA News Agency
4,Makan Bergizi Gratis,Menagih Mutu Pendidikan Setelah Disedot MBG - ...,"Sat, 02 May 2026 09:33:00 GMT",Kompas.id,https://www.kompas.id,https://news.google.com/rss/articles/CBMifkFVX...,https://www.kompas.id/artikel/menagih-mutu-pen...,Program Makan Bergizi Gratis (MBG) yang digada...,Stephanus Aranditio - stephanus.aranditio@komp...
...,...,...,...,...,...,...,...,...,...
595,Perpres Nomor 115 Tahun 2025,"BGN: Semua Guru, Tenaga Kebersihan hingga Staf...","Thu, 08 Jan 2026 08:00:00 GMT",Liputan6.com,https://www.liputan6.com,https://news.google.com/rss/articles/CBMitwFBV...,https://www.liputan6.com/news/read/6252989/bgn...,"Liputan6.com, Jakarta -Badan Gizi Nasional (BG...",Liputan6.com
596,Perpres Nomor 115 Tahun 2025,"BGN Larang SPPG Tolak Produk UMKM, Petani, dan...","Tue, 27 Jan 2026 08:00:00 GMT",JPNN.com Jatim,https://jatim.jpnn.com,https://news.google.com/rss/articles/CBMipAFBV...,https://jatim.jpnn.com/jatim-terkini/42922/bgn...,"jatim.jpnn.com, BONDOWOSO - Setiap Satuan Pela...",
597,Perpres Nomor 115 Tahun 2025,Dapur MBG Dilarang Tolak Pasokan UMKM hingga P...,"Wed, 28 Jan 2026 08:00:00 GMT","detikFinance - Berita Ekonomi Bisnis, dan Inve...",https://finance.detik.com,https://news.google.com/rss/articles/CBMiwAFBV...,https://finance.detik.com/berita-ekonomi-bisni...,Badan Gizi Nasional (BGN) menegaskan Satuan Pe...,Retno Ayuningrum
598,Perpres Nomor 115 Tahun 2025,"Anak Jalanan hingga Lansia Bakal dapat MBG, Be...","Sat, 06 Dec 2025 08:00:00 GMT",Tempo.co,https://www.tempo.co,https://news.google.com/rss/articles/CBMioAFBV...,https://www.tempo.co/politik/anak-jalanan-hing...,BADAN Gizi Nasional atau BGN mengungkapkan ana...,
